In [24]:
import requests
import numpy as np
import pandas as pd 
import pprint
import sqlite3
import json
from datetime import datetime

In [25]:
def create_database():
    conn = sqlite3.connect("macro_ml.db")
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS macro_data (
        date TEXT,
        indicator TEXT,
        value REAL,
        PRIMARY KEY (date, indicator)
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS price_data (
            date TEXT,
            ticker TEXT,
            adjusted_close REAL,
            volume INTEGER,
            PRIMARY KEY (date, ticker) 
        )
    """)

    conn.commit()
    conn.close()
    print("Database and tables initialized!")

def list_tables():
    conn = sqlite3.connect("macro_ml.db")
    cursor = conn.cursor()
    cursor.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type='table'
        AND name NOT LIKE 'sqlite_%'; 
    """)

    tables = cursor.fetchall()
    conn.close()

    print(tables)
    print("---------------")
    print("Tables in macro_ml.db:")
    for table in tables:
        print(f"- {table[0]}")

In [4]:
create_database()
list_tables()

Database and tables initialized!
[('macro_data',), ('price_data',)]
---------------
Tables in macro_ml.db:
- macro_data
- price_data


In [26]:
API_KEY = "AYBALAEUTJH8V4VW"
URL_SPY = f'https://www.alphavantage.co/query?function=TIME_SERIES_MONTHLY&symbol=SPY&apikey={API_KEY}'
SPY_DATA = requests.get(URL_SPY).json()

In [27]:
SPY_time_series = SPY_DATA.get("Monthly Time Series")

In [28]:
SPY_input = []
for date, monthly_data in SPY_time_series.items():
    price = float(monthly_data['4. close'])
    volume = int(monthly_data['5. volume'])
    SPY_input.append((date, 'SPY', price, volume))

In [29]:
with sqlite3.connect("macro_ml.db") as conn:

    cursor = conn.cursor() 
    cursor.executemany("""
        INSERT OR REPLACE INTO price_data (date, ticker, adjusted_close, volume)
        VALUES(?,?,?,?)
    """, SPY_input)
conn.close()

with sqlite3.connect("macro_ml.db") as conn:
    SPY_df = pd.read_sql("SELECT * FROM price_data", conn)
conn.close()

In [30]:
URL_CPI = f'https://www.alphavantage.co/query?function=CPI&interval=monthly&apikey={API_KEY}'
CPI_DATA = requests.get(URL_CPI)

In [31]:
CPI_data_monthly = CPI_DATA.json().get("data")

CPI_comp = []

for monthly in CPI_data_monthly:
    date_str = monthly['date']
    raw_value = monthly['value']
    CPI = float(raw_value) if raw_value != "." else None 
    converted_date = datetime.strptime(date_str, "%Y-%m-%d")
    if converted_date > datetime(2000, 1, 1):
        CPI_comp.append((date_str, "CPI", CPI))

In [32]:
with sqlite3.connect("macro_ml.db") as conn:

    cursor = conn.cursor()
    cursor.executemany("""
        INSERT OR REPLACE INTO macro_data (date, indicator, value)
        VALUES(?, ?, ?)
    """, CPI_comp)
conn.close()

with sqlite3.connect("macro_ml.db") as conn:
    CPI_df = pd.read_sql("SELECT * FROM macro_data WHERE indicator='CPI'", conn)
conn.close()

In [33]:
URL_UNEMPLOYMENT = 'https://www.alphavantage.co/query?function=UNEMPLOYMENT&apikey={API_KEY}'
UNEMPLOYMENT = requests.get(URL_UNEMPLOYMENT)

In [34]:
UNEMPLOYMENT_monthly = UNEMPLOYMENT.json().get("data")

UNEMP_comp = []

for unemp in UNEMPLOYMENT_monthly:
    date_str = unemp['date']
    raw_val = unemp['value']
    monthly_unemp = float(raw_val) if raw_val != "." else None
    converted_date = datetime.strptime(date_str, "%Y-%m-%d")
    if converted_date > datetime(2000,1,1):
        UNEMP_comp.append((date_str, "UNEMP", monthly_unemp))

In [35]:
with sqlite3.connect("macro_ml.db") as conn:
    cursor = conn.cursor()
    cursor.executemany("""
        INSERT OR REPLACE INTO macro_data (date, indicator, value)
        VALUES(?, ?, ?)
    """, UNEMP_comp)

conn.close()

with sqlite3.connect("macro_ml.db") as conn:
    UNEMP_pd = pd.read_sql("SELECT * FROM macro_data WHERE indicator='UNEMP'", conn)
conn.close()

print(UNEMP_pd)

           date indicator  value
0    2026-08-01     UNEMP    4.1
1    2026-07-01     UNEMP    4.1
2    2026-06-01     UNEMP    4.2
3    2026-05-01     UNEMP    4.3
4    2026-04-01     UNEMP    4.3
..          ...       ...    ...
314  2000-06-01     UNEMP    4.0
315  2000-05-01     UNEMP    4.0
316  2000-04-01     UNEMP    3.8
317  2000-03-01     UNEMP    4.0
318  2000-02-01     UNEMP    4.1

[319 rows x 3 columns]


In [36]:
URL_FRR = 'https://www.alphavantage.co/query?function=FEDERAL_FUNDS_RATE&interval=monthly&apikey={API_KEY}'
FRR = requests.get(URL_FRR).json()

In [64]:
FRR_monthly_data = FRR.get("data")

FRR_compiled = []

for month in FRR_monthly_data[:]:
    date_str_org = month['date']
    converted_date = datetime.strptime(date_str_org, "%Y-%m-%d")
    raw_val = float(monthly['value'])
    rate = raw_val if raw_val != "." else None 
    
    if converted_date >= datetime(2000,1,1):
        FRR_compiled.append((date_str_org, "Fed_Rate", rate))    

In [67]:
with sqlite3.connect("macro_ml.db") as conn:
    cursor = conn.cursor()
    cursor.executemany("""
        INSERT OR REPLACE INTO macro_data (date, indicator, value)
        VALUES(?, ?, ?)
    """, FRR_compiled)
conn.close() 

In [69]:
URL_10yr = url = 'https://www.alphavantage.co/query?function=TREASURY_YIELD&interval=monthly&maturity=10year&apikey={API_key}'
T10yr = requests.get(URL_10yr).json()

In [74]:
T10yr_monthly = T10yr.get("data")
T10yr_compiled = []

for month in T10yr_monthly:
    date_str = month['date']
    raw_val = month['value']
    rate = raw_val if raw_val != "." else None
    if date_str >= "2000-01-01":
        T10yr_compiled.append((date_str, "10yr", rate))        

In [75]:
with sqlite3.connect("macro_ml.db") as conn:
    cursor = conn.cursor()
    cursor.executemany("""
        INSERT OR REPLACE INTO macro_data (date, indicator, value)
        VALUES(?, ?, ?) 
    """, T10yr_compiled)
conn.close()

In [79]:
URL_RETAIL = 'https://www.alphavantage.co/query?function=RETAIL_SALES&apikey={API_Key}'
RETAIL_SALES = requests.get(URL_RETAIL).json()

In [86]:
monthly_retail = RETAIL_SALES.get("data")
retail_compiled = []

for month in monthly_retail:
    date_str = month['date']
    rate = month['value']
    if date_str >= "2000-01-01":
        retail_compiled.append((date_str, "retail", rate))

In [90]:
with sqlite3.connect("macro_ml.db") as conn:
    cursor = conn.cursor()
    cursor.executemany("""
        INSERT OR REPLACE INTO macro_data (date, indicator, value)
        VALUES(?, ?, ?) 
    """, retail_compiled)
conn.close()

In [91]:
### the macro_data table in SQL

with sqlite3.connect("macro_ml.db") as conn:
    total_macro_pd = pd.read_sql("SELECT* FROM macro_data", conn)

print(len(total_macro_pd))
unique_indicators = pd.unique(total_macro_pd['indicator'])
print(unique_indicators)

1598
<ArrowStringArray>
['CPI', 'UNEMP', 'Fed_Rate', '10yr', 'retail']
Length: 5, dtype: str
